## Standalone decoder comparison

The original Meta-TL pipeline above is unchanged. The next cell trains a separate decoder on the same fixed trials and preprocessed EMG. Its architecture retains the supplied model's spatial-mix, causal-TCN, and LSTM core, adapted to the same five-class temporal output, active-bin loss, and trial decision rule used by Meta-TL.


In [ ]:
# ============================================================
# STANDALONE COMPARISON — RAW UTAH EMG -> TCN -> LSTM -> GESTURE
# ============================================================

import csv
import copy
import random

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader


# The comparison intentionally reuses the exact same preprocessed Dataset
# objects and fixed splits created above. The Meta-TL run is unchanged.
STANDALONE_EPOCHS = AO_EPOCHS
STANDALONE_LR = AO_LR
STANDALONE_WEIGHT_DECAY = AO_WEIGHT_DECAY
STANDALONE_GRAD_CLIP_NORM = AO_GRAD_CLIP_NORM
STANDALONE_TCN_CHANNELS = (32, 64, 64)
STANDALONE_KERNEL_SIZE = 5
STANDALONE_DROPOUT = 0.20
STANDALONE_LSTM_HIDDEN = 64


def standalone_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


standalone_seed_everything(SEED)


# A fresh seeded loader makes the standalone run independent of the generator
# state consumed by the Meta-TL run.
standalone_train_generator = torch.Generator().manual_seed(SEED)
standalone_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    generator=standalone_train_generator,
)


@torch.no_grad()
def fit_standalone_channel_normalization(dataset, batch_size=BATCH_SIZE):
    """Fit per-channel mean/std using TRAIN EMG only."""
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    sums = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    sums_sq = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    count = 0

    for emg, _, _, _, _ in loader:
        emg = emg.to(torch.float64)
        sums += emg.sum(dim=(0, 2))
        sums_sq += emg.square().sum(dim=(0, 2))
        count += emg.shape[0] * emg.shape[2]

    mean = sums / count
    variance = (sums_sq / count - mean.square()).clamp_min(0.0)
    std = variance.sqrt().clamp_min(1e-6)
    return (
        mean.float().view(1, YOUR_CHANNELS, 1),
        std.float().view(1, YOUR_CHANNELS, 1),
    )


class StandaloneResidualBlock(nn.Module):
    """Causal two-convolution residual block adapted from the supplied model."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.downsample(x)
        out = self.conv1(F.pad(x, (self.left_pad, 0)))
        out = F.relu(out)
        out = self.conv2(F.pad(out, (self.left_pad, 0)))
        return F.relu(out + residual)


class StandaloneEmgDecoder(nn.Module):
    """
    Train-fitted normalization -> 1x1 spatial mixing -> causal TCN ->
    LSTM sequence -> five temporal gesture logits at the exact Meta output bins.

    Gram-Schmidt selection and pose PCA are deliberately omitted because they
    belong to the supplied 42-output pose-regression task, not this classifier.
    """
    def __init__(self, emg_mean, emg_std):
        super().__init__()
        self.register_buffer("emg_mean", emg_mean.clone().float())
        self.register_buffer("emg_std", emg_std.clone().float())
        self.spatial_mix = nn.Conv1d(YOUR_CHANNELS, YOUR_CHANNELS, 1)

        blocks = []
        in_channels = YOUR_CHANNELS
        for block_index, out_channels in enumerate(STANDALONE_TCN_CHANNELS):
            blocks.append(
                StandaloneResidualBlock(
                    in_channels,
                    out_channels,
                    STANDALONE_KERNEL_SIZE,
                    dilation=2 ** block_index,
                )
            )
            in_channels = out_channels
        self.tcn = nn.Sequential(*blocks)
        self.dropout = nn.Dropout(STANDALONE_DROPOUT)
        self.layer_norm = nn.LayerNorm(
            STANDALONE_TCN_CHANNELS[-1],
            elementwise_affine=False,
        )
        self.lstm = nn.LSTM(
            STANDALONE_TCN_CHANNELS[-1],
            STANDALONE_LSTM_HIDDEN,
            batch_first=True,
        )
        self.classifier = nn.Linear(STANDALONE_LSTM_HIDDEN, AO_KEPT_CLASSES)
        self.register_buffer(
            "output_indices",
            torch.arange(
                META_LEFT_CONTEXT,
                META_INPUT_SAMPLES,
                META_OUTPUT_STRIDE,
                dtype=torch.long,
            ),
        )

    def forward(self, emg):
        x = (emg - self.emg_mean) / self.emg_std
        x = self.spatial_mix(x)
        x = self.tcn(x)
        x = self.dropout(x)
        x = self.layer_norm(x.transpose(1, 2))
        sequence, _ = self.lstm(x)
        selected_sequence = sequence.index_select(1, self.output_indices)
        return self.classifier(selected_sequence).transpose(1, 2).contiguous()


standalone_emg_mean, standalone_emg_std = (
    fit_standalone_channel_normalization(train_dataset)
)
standalone_model = StandaloneEmgDecoder(
    standalone_emg_mean,
    standalone_emg_std,
).to(DEVICE)

standalone_parameter_count = sum(
    parameter.numel()
    for parameter in standalone_model.parameters()
    if parameter.requires_grad
)
meta_adapter_parameter_count = sum(
    parameter.numel()
    for parameter in model.adapter.parameters()
    if parameter.requires_grad
)

print("Standalone trainable parameters:", f"{standalone_parameter_count:,}")
print("Meta-TL adapter trainable parameters:", f"{meta_adapter_parameter_count:,}")
print("Both models use the same fixed trials, EMG preprocessing, batch size,")
print("epoch count, LR schedule, active-bin supervision, trial prediction rule,")
print("and validation selection rule.")
print("Base LR: Meta-TL", AO_LR, "| standalone", STANDALONE_LR)


standalone_optimizer = torch.optim.AdamW(
    build_adamw_parameter_groups(
        standalone_model,
        STANDALONE_WEIGHT_DECAY,
    ),
    lr=STANDALONE_LR,
)
def standalone_scheduled_lr(epoch_number):
    # Same warmup/decay shape as Meta-TL, but based on the standalone LR.
    if epoch_number <= AO_WARMUP_EPOCHS:
        fraction = epoch_number / max(1, AO_WARMUP_EPOCHS)
        return STANDALONE_LR * fraction
    if epoch_number >= AO_DECAY_EPOCH:
        return STANDALONE_LR * AO_DECAY_FACTOR
    return STANDALONE_LR


def active_only_multiclass_ce(logits, target, valid_mask):
    """Five-way cross-entropy evaluated only at active valid output bins."""
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid
    if int(keep.sum().item()) == 0:
        raise RuntimeError("Batch contains no active valid target bins.")

    true_classes = target.argmax(dim=1)
    logits_time_classes = logits.transpose(1, 2).contiguous()
    return F.cross_entropy(
        logits_time_classes[keep],
        true_classes[keep],
    )


def run_standalone_epoch(loader, train):
    standalone_model.train(train)
    loss_sum = 0.0
    batch_count = 0
    correct = 0
    trial_count = 0
    gradient_norm_sum = 0.0
    gradient_step_count = 0

    for emg, target, valid_mask, gestures, _ in loader:
        emg = emg.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        valid_mask = valid_mask.to(DEVICE, non_blocking=True)
        gestures = gestures.to(DEVICE, non_blocking=True)

        if train:
            standalone_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = standalone_model(emg)
            target, valid_mask = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )
            loss = active_only_multiclass_ce(
                logits,
                target,
                valid_mask,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite standalone loss detected.")

            if train:
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    standalone_model.parameters(),
                    STANDALONE_GRAD_CLIP_NORM,
                )
                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Non-finite standalone gradient norm detected."
                    )
                if float(gradient_norm.detach().item()) <= 0.0:
                    raise RuntimeError(
                        "Standalone gradient norm is zero; backpropagation is disconnected."
                    )
                standalone_optimizer.step()
                gradient_norm_sum += float(gradient_norm.detach().item())
                gradient_step_count += 1

        batch_results = trial_predictions(
            logits.detach(),
            target.detach(),
            valid_mask.detach(),
            gestures.detach(),
        )
        correct += sum(
            int(result["prediction"] == result["true"])
            for result in batch_results
        )
        trial_count += len(batch_results)
        loss_sum += float(loss.detach().item())
        batch_count += 1

    return {
        "loss": loss_sum / max(1, batch_count),
        "accuracy": correct / max(1, trial_count),
        "num_trials": trial_count,
        "gradient_norm": (
            gradient_norm_sum / gradient_step_count
            if gradient_step_count
            else float("nan")
        ),
    }


standalone_history = {
    "learning_rate": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_gradient_norm": [],
}
standalone_best_state = None
standalone_best_epoch = None
standalone_best_val_accuracy = float("-inf")
standalone_best_val_loss = float("inf")

print("\nStarting standalone decoder training.")
for epoch in range(1, STANDALONE_EPOCHS + 1):
    current_lr = standalone_scheduled_lr(epoch)
    set_optimizer_lr(standalone_optimizer, current_lr)

    train_metrics = run_standalone_epoch(standalone_train_loader, train=True)
    val_metrics = run_standalone_epoch(val_loader, train=False)

    standalone_history["learning_rate"].append(current_lr)
    standalone_history["train_loss"].append(train_metrics["loss"])
    standalone_history["val_loss"].append(val_metrics["loss"])
    standalone_history["train_accuracy"].append(train_metrics["accuracy"])
    standalone_history["val_accuracy"].append(val_metrics["accuracy"])
    standalone_history["train_gradient_norm"].append(
        train_metrics["gradient_norm"]
    )

    print(
        f"Standalone epoch {epoch:03d}/{STANDALONE_EPOCHS} | "
        f"lr {current_lr:.2e} | "
        f"train CE {train_metrics['loss']:.4f} | "
        f"val CE {val_metrics['loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.3f} | "
        f"val acc {val_metrics['accuracy']:.3f} | "
        f"grad {train_metrics['gradient_norm']:.3e}"
    )

    is_better = (
        val_metrics["accuracy"] > standalone_best_val_accuracy
        or (
            np.isclose(
                val_metrics["accuracy"],
                standalone_best_val_accuracy,
            )
            and val_metrics["loss"] < standalone_best_val_loss
        )
    )
    if is_better:
        standalone_best_epoch = epoch
        standalone_best_val_accuracy = val_metrics["accuracy"]
        standalone_best_val_loss = val_metrics["loss"]
        standalone_best_state = copy.deepcopy(standalone_model.state_dict())


if standalone_best_state is None:
    raise RuntimeError("No standalone best state was recorded.")

standalone_model.load_state_dict(standalone_best_state)
standalone_model.to(DEVICE)
standalone_test_metrics = run_standalone_epoch(test_loader, train=False)

print("\nRestored standalone epoch", standalone_best_epoch)
print("Standalone best validation accuracy:", standalone_best_val_accuracy)
print("Standalone best validation active-bin CE:", standalone_best_val_loss)
print("Standalone blind test active-bin CE:", standalone_test_metrics["loss"])
print("Standalone blind test accuracy:", standalone_test_metrics["accuracy"])


In [ ]:
# ============================================================
# DIRECT META-TL VS STANDALONE LEARNING-CURVE COMPARISON
# ============================================================

COMPARISON_EXPORT_DIR = RESULTS_EXPORT_DIR / "meta_vs_standalone"
COMPARISON_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

comparison_epochs = np.arange(1, AO_EPOCHS + 1)
meta_learning_rate = np.asarray(
    [scheduled_lr(epoch) for epoch in comparison_epochs],
    dtype=float,
)

comparison_figure, axes = plt.subplots(
    2,
    2,
    figsize=(12.0, 8.0),
    sharex=True,
)

axes[0, 0].plot(
    comparison_epochs,
    history["train_task_loss"],
    label="Train",
    linewidth=2,
)
axes[0, 0].plot(
    comparison_epochs,
    history["val_task_loss"],
    label="Validation",
    linewidth=2,
)
axes[0, 0].set_title("Meta-TL active-bin BCE")
axes[0, 0].set_ylabel("Loss")

axes[0, 1].plot(
    comparison_epochs,
    standalone_history["train_loss"],
    label="Train",
    linewidth=2,
)
axes[0, 1].plot(
    comparison_epochs,
    standalone_history["val_loss"],
    label="Validation",
    linewidth=2,
)
axes[0, 1].set_title("Standalone active-bin multiclass CE")
axes[0, 1].set_ylabel("Loss")

axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(history["train_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["train_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 0].set_title("Training trial accuracy")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylim(-2, 102)

axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(history["val_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["val_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 1].set_title("Validation trial accuracy")
axes[1, 1].set_ylabel("Accuracy (%)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylim(-2, 102)

for axis in axes.reshape(-1):
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)

comparison_figure.suptitle(
    "Matched data and evaluation: frozen Meta transfer vs standalone decoder",
    fontsize=13,
)
comparison_figure.tight_layout()
save_figure_adobe(
    comparison_figure,
    COMPARISON_EXPORT_DIR,
    "meta_tl_vs_standalone_learning_curves",
)
plt.show()


lr_figure, lr_axis = plt.subplots(figsize=(8.0, 3.6))
lr_axis.plot(
    comparison_epochs,
    meta_learning_rate,
    color="tab:blue",
    linewidth=2,
    label="Meta-TL",
)
lr_axis.plot(
    comparison_epochs,
    standalone_history["learning_rate"],
    color="tab:orange",
    linewidth=2,
    label="Standalone",
)
lr_axis.set_title("Shared learning-rate schedule")
lr_axis.set_xlabel("Epoch")
lr_axis.set_ylabel("Learning rate")
lr_axis.set_yscale("log")
lr_axis.grid(alpha=0.25)
lr_axis.legend(frameon=False)
lr_figure.tight_layout()
save_figure_adobe(
    lr_figure,
    COMPARISON_EXPORT_DIR,
    "shared_learning_rate_schedule",
)
plt.show()


history_csv_path = COMPARISON_EXPORT_DIR / "meta_tl_vs_standalone_history.csv"
with history_csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "epoch",
        "meta_learning_rate",
        "standalone_learning_rate",
        "meta_train_bce",
        "meta_val_bce",
        "meta_train_accuracy",
        "meta_val_accuracy",
        "standalone_train_active_bin_ce",
        "standalone_val_active_bin_ce",
        "standalone_train_accuracy",
        "standalone_val_accuracy",
    ])
    for index, epoch in enumerate(comparison_epochs):
        writer.writerow([
            int(epoch),
            float(meta_learning_rate[index]),
            float(standalone_history["learning_rate"][index]),
            float(history["train_task_loss"][index]),
            float(history["val_task_loss"][index]),
            float(history["train_accuracy"][index]),
            float(history["val_accuracy"][index]),
            float(standalone_history["train_loss"][index]),
            float(standalone_history["val_loss"][index]),
            float(standalone_history["train_accuracy"][index]),
            float(standalone_history["val_accuracy"][index]),
        ])

standalone_checkpoint_path = (
    COMPARISON_EXPORT_DIR / "best_standalone_emg_decoder.pt"
)
torch.save(
    {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in standalone_model.state_dict().items()
        },
        "best_epoch": standalone_best_epoch,
        "best_val_accuracy": standalone_best_val_accuracy,
        "best_val_loss": standalone_best_val_loss,
        "test_metrics": standalone_test_metrics,
        "history": standalone_history,
        "class_names": AO_CLASS_NAMES,
        "data_path": str(DATA_PATH),
        "architecture": {
            "kind": "normalized-spatial-mix-causal-tcn-lstm-temporal-classifier",
            "input_channels": YOUR_CHANNELS,
            "input_samples": META_INPUT_SAMPLES,
            "tcn_channels": STANDALONE_TCN_CHANNELS,
            "kernel_size": STANDALONE_KERNEL_SIZE,
            "dropout": STANDALONE_DROPOUT,
            "lstm_hidden": STANDALONE_LSTM_HIDDEN,
            "output_classes": AO_KEPT_CLASSES,
            "output_bins": EXPECTED_META_OUTPUT_SAMPLES,
        },
    },
    standalone_checkpoint_path,
)

print("Comparison history:", history_csv_path.resolve())
print("Standalone checkpoint:", standalone_checkpoint_path.resolve())
print()
print("Meta-TL loss is active-bin BCE; standalone loss is active-bin multiclass CE.")
print("Losses are plotted separately because their numerical scales differ.")
print("Both accuracy curves use the same active-bin mean-logit trial decision.")
